# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuan05-afk/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.800
Precision@50:  hand rule 0.680   vs   tree 0.740


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

### Yuan's Your-turn notes (executed)

**What I tried.** Depth-3 decision tree on non-leaky numeric features only (`trend_direction` / `trend_pct` excluded). Compared in-sample precision to a **client-holdout** split (`GroupShuffleSplit` on `client_id`, test_size=0.2, seed=42), then reported Precision@20 and Precision@50 on held-out clients.

**Learning.** In-sample scores flatter the model. Client-holdout is the honest bar for a multi-tenant SEO warehouse: a client's pages must not appear in both train and test. If Precision@K collapses on unseen clients, the model is not ready for a refresh queue that will be read by editors.

**Product takeaway.** Keep the hand-rule baseline as the transparent floor; promote a tree/RF only when holdout Precision@K beats that floor by a margin that justifies review cost.


### Yuan's Your-turn notes (executed)

**What I tried.** Depth-3 balanced tree on non-leaky features from `df`
(`impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`,
`word_count`, `content_age_days`, `days_since_last_update`). Compared in-sample
precision to a **client-holdout** split (`GroupShuffleSplit` on `client_id`,
test_size=0.2, seed=42), then reported Precision@20 / @50 on held-out clients.

**Measured (this run):** In-sample Precision@all ≈ **0.651**; client-holdout
Precision@all ≈ **0.579** (7 holdout clients, 6163 pages); holdout
**Precision@20 ≈ 0.700**, **Precision@50 ≈ 0.680**. The tree first splits on
`impressions_90d`, then `content_age_days` / `ctr` / `avg_position`.

**Learning.** In-sample flattens the story; holdout is the honest bar. Precision@K
on held-out clients stays useful, but the gap vs in-sample is the reminder that
client mix can inflate teaching scores.

**Product takeaway.** Keep the hand rule as the readable baseline; promote the
tree only when holdout Precision@K beats it without leaking trend labels.



In [7]:
# Your turn — depth / feature try + honest client-holdout check
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import precision_score
import numpy as np
import pandas as pd

# Build from df (not the earlier X frame) so we can try engagement_rate, etc.
# Never include trend_direction / trend_pct — those leak the declining label.
_candidates = [
    "impressions_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate",
    "word_count", "content_age_days", "days_since_last_update",
]
feat_cols = [c for c in _candidates if c in df.columns]
X_use = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0).reset_index(drop=True)
y_use = pd.Series(df["is_declining_label"].to_numpy().astype(int)).reset_index(drop=True)
groups = pd.Series(df["client_id"].to_numpy()).reset_index(drop=True)
print("Using features:", feat_cols)

# In-sample depth-3 tree (teaching contrast only)
tree_in = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_in.fit(X_use, y_use)
print(f"In-sample Precision@all (depth-3): {precision_score(y_use, tree_in.predict(X_use)):.3f}")
print(export_text(tree_in, feature_names=feat_cols))

# Client-holdout: train on ~80% of clients, score on unseen clients
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X_use, y_use, groups))
tree_ho = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_ho.fit(X_use.iloc[tr_idx], y_use.iloc[tr_idx])
proba = tree_ho.predict_proba(X_use.iloc[te_idx])[:, 1]
y_te = y_use.iloc[te_idx]
print(f"Client-holdout Precision@all (depth-3): {precision_score(y_te, tree_ho.predict(X_use.iloc[te_idx])):.3f}")
print(f"Holdout clients={groups.iloc[te_idx].nunique()} pages={len(te_idx)}")
order = np.argsort(-proba)
for k in (20, 50):
    print(f"Holdout Precision@{k}: {y_te.iloc[order[:k]].mean():.3f}")
print(
    "Learning: if holdout Precision@K is below the hand rule or collapses vs in-sample, "
    "the tree was memorizing client mix; keep client-aware splits in the real pipeline."
)


Using features: ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'word_count', 'content_age_days', 'days_since_last_update']
In-sample Precision@all (depth-3): 0.651
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0

Client-holdout Precision@all (depth-3): 0.579
Holdout clients=7 pages=6163
Holdout Precision@20: 0.700
Holdout Precision@50: 0.680
Le

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.